<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pipeline_Prune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline_Prune — MobileNetV2 Structured 10%

Exports `mobilenetv2_pruned_structured_10pct.pth` → FP32 ONNX + INT8 QDQ ONNX for STM32 deployment,
then prints a head-to-head comparison against the `mobilenetv2_seed_74` baseline from `Pipeline_Quantz`.

**What this notebook does:**
1. Loads the pruned checkpoint (masks already removed by `Model_Pruning`)
2. Reports val accuracy and actual sparsity
3. Exports FP32 ONNX + INT8 QDQ ONNX
4. Saves test files for STM32 on-device evaluation
5. Prints a final comparison table: Baseline vs Pruned (FP32 and INT8)

> **Baseline results** come from `Pipeline_Quantz` — they are read directly
> from `exports/mobilenetv2_seed_74/` with no re-running needed.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

!pip -q install onnx onnxruntime onnxruntime-tools
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORTS  = Path("/content/drive/My Drive/stm32-thesis/exports")

# ── Pruned model ──
PRUNED_CKPT = f"{SAVE_DIR}/mobilenetv2_pruned_structured_10pct.pth"
PRUNED_DIR  = EXPORTS / "mobilenetv2_pruned_structured_10pct"
PRUNED_DIR.mkdir(parents=True, exist_ok=True)

# ── Baseline (mobilenetv2_seed_74 from Pipeline_Quantz) ──
# These files already exist — no re-running Pipeline_Quantz needed.
BASELINE_DIR        = EXPORTS / "mobilenetv2_seed_74"
BASELINE_LABELS_NPZ = BASELINE_DIR / "test_labels.npz"

# Verify paths before continuing
checks = {
    "Pruned checkpoint" : Path(PRUNED_CKPT),
    "Baseline labels"   : BASELINE_LABELS_NPZ,
}
all_ok = True
for label, p in checks.items():
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"  {status}  {label}: {p}")
    if not p.exists():
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Fix the missing paths above before continuing.")

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────

def model_sparsity(model: nn.Module) -> float:
    """Fraction of exactly-zero weights across all Conv2d + Linear layers."""
    zeros, total = 0, 0
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total > 0 else 0.0


def export_onnx(model: nn.Module, path: Path) -> None:
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    size_mb = path.stat().st_size / 1e6
    print(f"    ✅ FP32 ONNX: {path}  ({size_mb:.2f} MB)")


def save_test_files(model: nn.Module, test_loader, out_dir: Path, n: int = 200):
    """Save input / logits / labels from TEST split for STM32 on-device eval."""
    inputs, logits_list, labels = [], [], []
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            if i >= n:
                break
            x = x.to(device)
            out = model(x)
            inputs.append(x.cpu().numpy().astype("float32")[0])
            logits_list.append(out.cpu().numpy().astype("float32")[0])
            labels.append(int(y.item()))
    inputs = np.stack(inputs)
    labels = np.array(labels, dtype="int32")
    logits = np.stack(logits_list)
    np.savez(out_dir / "test_input.npz",  input=inputs)
    np.savez(out_dir / "test_labels.npz", label=labels)
    np.savez(out_dir / "test_logits.npz", input=inputs, logits=logits)
    print(f"    ✅ Test files saved — inputs: {inputs.shape}  labels: {labels.shape}")
    return out_dir / "test_input.npz", out_dir / "test_labels.npz"


def save_calib_file(train_loader, path: Path, n: int = 200) -> None:
    """Calibration data from TRAIN split only — never test or val."""
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n:
                break
            xs.append(x.numpy().astype("float32")[0])
    xs = np.stack(xs)
    np.savez(path, input=xs)
    print(f"    ✅ Calib data saved: {path}  {xs.shape}")


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data):
            return None
        out = {"input": self.data[self.i : self.i + 1]}
        self.i += 1
        return out
    def rewind(self):
        self.i = 0


def quantize_int8(fp32_path: Path, calib_path: Path, int8_path: Path) -> None:
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    size_mb = int8_path.stat().st_size / 1e6
    print(f"    ✅ INT8 QDQ ONNX: {int8_path}  ({size_mb:.2f} MB)")


def compute_stm32_accuracy(
    labels_npz: Path,
    outputs_npz: Path,
    key: str = "c_outputs_1",
    num_classes: int = 2,
) -> float:
    labels = np.load(labels_npz)["label"].astype("int64")
    raw    = np.load(outputs_npz)[key]
    if raw.size != len(labels) * num_classes:
        raise ValueError(f"Size mismatch: {raw.size} vs {len(labels) * num_classes}")
    preds = np.argmax(raw.reshape(len(labels), num_classes), axis=1)
    return float((preds == labels).mean() * 100)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64)
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

In [ ]:
# ── Load pruned model ────────────────────────────────────────────────
# Model_Pruning calls prune.remove() before saving, so the checkpoint
# is a plain state_dict — no mask buffers, no special loading needed.
model = VWW_MobileNetV2().to(device)
model.load_state_dict(torch.load(PRUNED_CKPT, map_location=device))
model.eval()

val_acc  = evaluate(model, val_loader, device)
sparsity = model_sparsity(model)

total_params    = sum(p.numel() for p in model.parameters())
nonzero_params  = sum((p != 0).sum().item() for p in model.parameters())

print(f"Val accuracy  : {val_acc * 100:.2f}%")
print(f"Sparsity      : {sparsity * 100:.2f}%  (zero weights in Conv2d + Linear)")
print(f"Total params  : {total_params:,}")
print(f"Non-zero params: {nonzero_params:,}")

In [ ]:
# ── Export FP32 ONNX ─────────────────────────────────────────────────
fp32_path = PRUNED_DIR / "model_fp32.onnx"
export_onnx(model, fp32_path)

In [ ]:
# ── Save test files (TEST split — same 200 samples as baseline) ──────
input_npz, labels_npz = save_test_files(model, test_loader, PRUNED_DIR, n=200)

In [ ]:
# ── Save calibration data (TRAIN split only) ─────────────────────────
calib_npz = PRUNED_DIR / "calib_train.npz"
save_calib_file(train_loader, calib_npz, n=200)

In [ ]:
# ── INT8 QDQ quantisation ────────────────────────────────────────────
int8_path = PRUNED_DIR / "model_int8_qdq.onnx"
quantize_int8(fp32_path, calib_npz, int8_path)

In [ ]:
# ── ONNX Runtime sanity check ────────────────────────────────────────
sess   = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
sample = np.load(input_npz)["input"][0:1]
out    = sess.run(["logits"], {"input": sample})[0]
print(f"✅ FP32 ONNX check — output shape: {out.shape}  logits: {out[0]}")

sess_int8 = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
out_int8  = sess_int8.run(["logits"], {"input": sample})[0]
print(f"✅ INT8 ONNX check — output shape: {out_int8.shape}  logits: {out_int8[0]}")

## Run on STM32

Run both models through the STM32 AI CLI:

```bash
# FP32 pruned
stedgeai validate -m exports/mobilenetv2_pruned_structured_10pct/model_fp32.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/mobilenetv2_pruned_structured_10pct/test_input.npz
cp network_val_io.npz exports/mobilenetv2_pruned_structured_10pct/output_fp32.npz

# INT8 pruned
stedgeai validate -m exports/mobilenetv2_pruned_structured_10pct/model_int8_qdq.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/mobilenetv2_pruned_structured_10pct/test_input.npz
cp network_val_io.npz exports/mobilenetv2_pruned_structured_10pct/output_int8.npz
```

The baseline STM32 outputs (`output_fp32.npz`, `output_int8.npz`) should already
exist at `exports/mobilenetv2_seed_74/` from the `Pipeline_Quantz` run.
If not, run those CLI commands too before executing the final cell.

In [ ]:
# ── Final comparison: Baseline vs Pruned on STM32 ────────────────────
# Reads STM32 CLI output files from both export directories.
# Prints 'pending' for any file not yet present.

def _read_acc(labels_npz, outputs_npz):
    """Returns (accuracy_str, raw_float_or_nan)."""
    if not Path(outputs_npz).exists():
        return "pending", float("nan")
    try:
        acc = compute_stm32_accuracy(labels_npz, outputs_npz)
        return f"{acc:.2f}%", acc
    except Exception as e:
        return f"error: {e}", float("nan")


# Baseline
bl_fp32_str, bl_fp32 = _read_acc(BASELINE_LABELS_NPZ,  BASELINE_DIR / "output_fp32.npz")
bl_int8_str, bl_int8 = _read_acc(BASELINE_LABELS_NPZ,  BASELINE_DIR / "output_int8.npz")

# Pruned
pr_fp32_str, pr_fp32 = _read_acc(labels_npz, PRUNED_DIR / "output_fp32.npz")
pr_int8_str, pr_int8 = _read_acc(labels_npz, PRUNED_DIR / "output_int8.npz")

# Deltas (pruned - baseline)
def _delta_str(a, b):
    return f"{a - b:+.2f}%" if (a == a and b == b) else "—"

fp32_delta = _delta_str(pr_fp32, bl_fp32)
int8_delta = _delta_str(pr_int8, bl_int8)

W = 36
print("\n" + "=" * 70)
print("  STM32 ON-DEVICE RESULTS — MobileNetV2: Baseline vs Pruned (10%)")
print("=" * 70)
print(f"  {'Model':<{W}} {'FP32':>8}  {'INT8':>8}")
print("  " + "-" * 66)
print(f"  {'MobileNetV2 baseline (seed 74)':<{W}} {bl_fp32_str:>8}  {bl_int8_str:>8}")
print(f"  {'MobileNetV2 pruned structured 10%':<{W}} {pr_fp32_str:>8}  {pr_int8_str:>8}")
print("  " + "-" * 66)
print(f"  {'Delta (pruned − baseline)':<{W}} {fp32_delta:>8}  {int8_delta:>8}")
print("=" * 70)

# Contextual note
if pr_int8 == pr_int8 and bl_int8 == bl_int8:  # neither is nan
    if pr_int8 >= bl_int8:
        print(f"\n✅ Pruning preserved INT8 accuracy (+{pr_int8 - bl_int8:.2f}%). "
              f"Smaller model at no accuracy cost on-device.")
    else:
        print(f"\n⚠️  INT8 accuracy dropped {bl_int8 - pr_int8:.2f}% after pruning. "
              f"Check whether fine-tuning recovered sufficiently for your deployment threshold.")